In [ ]:
import torch
import ultralytics
from ultralytics import YOLO

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version: {torch.__version__}")

gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
import yaml

yaml_path = "data gigi/data.yaml"
try:
    with open(yaml_path, "r") as file:
        data_config = yaml.safe_load(file)
    print("--- Konfigurasi Dataset ---")
    print(f"Train Path : {data_config.get('train')}")
    print(f"Val Path   : {data_config.get('val')}")
    print(f"Test Path  : {data_config.get('test')}")
    print(f"Jumlah Class (nc) : {data_config.get('nc')}")
    print(f"Nama Class        : {data_config.get('names')}")
except FileNotFoundError:
    print(f"Error: File '{yaml_path}' tidak ditemukan.")

In [ ]:
model = YOLO("yolov12n.pt")

print("=== MEMULAI PROSES TRAINING YOLOv12 (DETEKSI GIGI) ===")
results = model.train(
    data="data gigi/data.yaml",
    epochs=50,                       # 50 epoch cukup untuk testing awal
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    name="yolov12_test_gigi",
    save=True,
    plots=True
)

print("\nTraining Selesai!")
print("Model terbaik tersimpan di: runs/detect/yolov12_test_gigi/weights/best.pt")

In [ ]:
best_model_path = "runs/detect/yolov12_test_gigi/weights/best.pt"
model_val = YOLO(best_model_path)

print("=== MEMULAI EVALUASI MODEL ===")
metrics = model_val.val(
    data="data gigi/data.yaml",
    split="val",
    imgsz=640,
    device=0 if torch.cuda.is_available() else 'cpu'
)

print("\n--- Hasil Metrik Performa ---")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

test_images_path = "data gigi/test/image"

results = model_val.predict(
    source=test_images_path,
    conf=0.5,
    save=True,
    project="runs/detect",
    name="hasil_test_prediksi"
)

output_dir = "runs/detect/hasil_test_prediksi"
if os.path.exists(output_dir):
    predicted_files = [f for f in os.listdir(output_dir) if f.endswith(('.jpg', '.png', '.jpeg'))][:3]
    if predicted_files:
        fig, axes = plt.subplots(1, len(predicted_files), figsize=(15, 5))
        if len(predicted_files) == 1:
            axes = [axes]
        for ax, file_name in zip(axes, predicted_files):
            img = Image.open(os.path.join(output_dir, file_name))
            ax.imshow(img)
            ax.axis('off')
        plt.tight_layout()
        plt.show()